In [2]:
!sudo apt-get update && sudo apt-get install -y zstd

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]       
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,703 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,040 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,973 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/univers

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh



>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess
import threading
import time

import requests


def wait_for_ollama(timeout=60, interval=1):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            response = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
            if response.ok:
                return True
        except requests.RequestException:
            time.sleep(interval)
    return False


def launch_ollama():
    # Start Ollama as a background process and wait until it is reachable.
    def run():
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )

    threading.Thread(target=run, daemon=True).start()
    if not wait_for_ollama():
        raise RuntimeError("Ollama did not become ready on http://127.0.0.1:11434")
    print("Ollama is serving in the background!")


launch_ollama()

Ollama is serving in the background!


In [5]:
!ollama pull bge-m3

In [6]:
!pip install requests
!pip install pandas

In [7]:
import requests

In [8]:
import requests


def create_embeddings(text_list):
    # embeddings = []
    # for idx, text in enumerate(text_list):
    #     try:
    #         res = requests.post(
    #             "http://localhost:11434/api/embed",
    #             json={"model": "bge-m3", "input": [text]},
    #         )
    #         if res.status_code != 200:
    #             print(f"❌ Chunk at index {idx} failed! Content: {text[:100]}...")
    #             embeddings.append(0)
    #         else:
    #             embedding = res.json()["embeddings"]
    #             embeddings.append(embedding)
    #     except Exception as e:
    #         print(f"Error on index {idx}: {e}")

    # return embeddings
    response = requests.post("http://localhost:11434/api/embed", json={
        "model": "bge-m3",
        "input": text_list
    })
    
    embeddings = response.json()["embeddings"]
    return embeddings

In [21]:
# response = create_embeddings(["What is RAG-based AI?", "How does it work?"])
# print(response)

import os
import json
import pandas as pd
jsons = os.listdir("jsons")
my_dict = []
if not os.path.exists("json_embeddings"):
    os.makedirs("json_embeddings")
    
    
    for json_file in jsons:
        with open(os.path.join("jsons", json_file), "r") as f:
            content = json.load(f)
        text_list = [c["text"] for c in content["chunks"]] 
        embeddings = create_embeddings(text_list)

        for i,chunk in enumerate(content["chunks"]):
            chunk['embedding'] = embeddings[i]
            my_dict.append(chunk)

        with open(os.path.join("json_embeddings", json_file), "w") as f:
            json.dump(content, f)

    df = pd.DataFrame.from_records(my_dict)
    print(df)

else:
    print("Embeddings already created. Loading from json_embeddings folder.")
    jsons = os.listdir("json_embeddings")
    for json_file in jsons:
        with open(os.path.join("json_embeddings", json_file), "r") as f:
            content = json.load(f)
        for chunk in content["chunks"]:
            my_dict.append(chunk)

    df = pd.DataFrame.from_records(my_dict)
    print(df)

Embeddings already created. Loading from json_embeddings folder.
     number      title  start    end  \
0         0  fullStack    0.0    2.0   
1         1  fullStack    2.0    7.0   
2         2  fullStack    7.0   10.0   
3         3  fullStack   10.0   12.0   
4         4  fullStack   12.0   14.0   
..      ...        ...    ...    ...   
687     221        Api  404.6  406.6   
688     222        Api  406.6  407.6   
689     223        Api  407.6  409.6   
690     224        Api  409.6  411.6   
691     225        Api  411.6  412.6   

                                                  text  \
0              Dear students, welcome to GIT Smashers.   
1     On a daily basis, how many applications and w...   
2          If I look at all of them in a general term,   
3             then what can I say? I can say a system.   
4                        Because in this whole system,   
..                                                 ...   
687                          Who does all these 

In [22]:
df

,number,title,start,end,text,embedding
0,0,fullStack,0.0,2.0,"Dear students, welcome to GIT Smashers.","[-0.052201383, -0.033823553, -0.04482444, -0.0..."
1,1,fullStack,2.0,7.0,"On a daily basis, how many applications and w...","[-0.021668456, 0.00074398535, -0.03190032, -0...."
2,2,fullStack,7.0,10.0,"If I look at all of them in a general term,","[-0.0041631004, 0.015698684, -0.04435234, -0.0..."
3,3,fullStack,10.0,12.0,then what can I say? I can say a system.,"[0.0038906594, 0.04320339, -0.043508545, 0.022..."
4,4,fullStack,12.0,14.0,"Because in this whole system,","[0.0022435247, 0.038135797, -0.03948991, -0.00..."
...,...,...,...,...,...,...
687,221,Api,404.6,406.6,Who does all these works?,"[-0.0242926, 0.007857073, -0.03545657, -0.0103..."
688,222,Api,406.6,407.6,API Gateway does.,"[-0.027941555, -0.014906803, -0.022414332, 0.0..."
689,223,Api,407.6,409.6,Load all these points properly.,"[0.02820909, 0.0034405028, -0.04957148, -0.003..."
690,224,Api,409.6,411.6,Ask anywhere in interviews.,"[-0.035089042, 0.015403673, -0.038453165, 0.01..."


In [11]:
incoming_query = "where the api is used?"
question_embedding = create_embeddings([incoming_query])
print(question_embedding[0:10])

[[-5.7949856e-05, -0.02059665, -0.035967343, 0.03533328, -0.029846989, -0.06556572, 0.009191434, 0.01977783, 0.01356428, 0.007023014, -0.05140783, 0.011586406, 0.014295533, 0.007504994, -0.0063590487, 0.0007217731, 0.024216095, 0.0017875248, -0.007607412, -0.010505735, -0.01758394, 0.015210393, -0.029546412, 0.008712488, 0.01670464, 0.01730797, -0.05032576, 0.00837328, 0.015309044, 0.005937921, -0.023964234, -0.015576452, 0.00092144794, 0.027942408, -0.027736735, -0.008284922, -0.010995487, -0.0270495, -0.05595813, -0.0022647686, 0.014231114, -0.024902774, -0.003056445, 0.006520387, 0.01965123, -0.0338143, -0.011456748, -0.004275721, -0.020304693, 0.00092528854, -0.014948534, -0.012331547, 0.017128846, -0.034370806, -0.008827972, 0.039759975, -0.03759369, -0.008003687, -0.088830926, 0.014395527, -0.042031366, -0.025208544, -0.011260171, 0.036800288, -0.003797347, 0.028738478, -0.036463723, -0.014557042, -0.03657446, -0.022740295, -0.030692905, 0.018833052, -0.029987108, 0.031054817, -0

In [13]:
import numpy as np

In [23]:
print(np.vstack(df["embedding"].values))
print(np.vstack(df["embedding"].shape))

[[-0.05220138 -0.03382355 -0.04482444 ... -0.00819716 -0.07798567
  -0.018034  ]
 [-0.02166846  0.00074399 -0.03190032 ... -0.01623557 -0.02591734
   0.00977472]
 [-0.0041631   0.01569868 -0.04435234 ...  0.02127435 -0.01517825
   0.00109047]
 ...
 [ 0.02820909  0.0034405  -0.04957148 ... -0.00056302 -0.00200056
   0.02137061]
 [-0.03508904  0.01540367 -0.03845316 ... -0.0513599   0.03492808
   0.00658227]
 [-0.01733893  0.04523436 -0.0467768  ... -0.00430194 -0.01212464
  -0.0191449 ]]
[[692]]


In [24]:
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(
    np.vstack(df["embedding"].values),
    np.array(question_embedding)
).flatten()
top_results = 5
result_indx = similarities.argsort()[-top_results:][::-1]

new_df = df.loc[result_indx]
print(new_df)

     number      title  start    end  \
145     145  fullStack  299.0  300.0   
496      30        Api   64.6   65.9   
498      32        Api   67.9   70.4   
530      64        Api  128.6  131.6   
468       2        Api    3.8    7.3   

                                            text  \
145                       what is API basically?   
496                           So what is an API?   
498                API is a bridge through which   
530                   For that, the API is used.   
468   what is an API and what is an API gateway.   

                                             embedding  
145  [-0.016398286, -0.032791313, -0.010064738, 0.0...  
496  [-0.022630775, -0.021871246, -0.020021437, 0.0...  
498  [0.01238366, -0.020571945, -0.018237477, 0.017...  
530  [-0.03497815, -0.009744706, -0.047095977, 0.01...  
468  [-0.0049872017, -0.02862097, -0.013981763, 0.0...  


In [ ]:
import os

def stop_ollama():
    # Sends a termination signal to any running ollama process
    os.system("pkill ollama")
    print(" Ollama server has been stopped.")

# stop_ollama()

 Ollama server has been stopped.
